# Toy representations

Representation geometry in dot and sprite worlds. The analysis compares
physical coordinates with the learned embeddings and examines how the
PCA spectrum relates to controllable degrees of freedom.

## Run

Select a completed run below; see the [experiment instructions](README.md).
`dot_smoke` is the one-epoch example, while `dot` uses the full configuration.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "toy/train.py").is_file())
TOY = ROOT / "toy"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(TOY))

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import torch
from torch.utils.data import DataLoader
import yaml
from sklearn.decomposition import PCA
from train import build_world, evaluation_seed
from models import CNNEncoder, CNNDecoder
from planning.plot_style import FULL_WIDTH, BLUE, RED, apply_matplotlib_style, position_colors

apply_matplotlib_style()
RUN = "dot"  # dot, structured, sprite, or their _smoke runs
DIRECTORY = TOY / "outputs" / RUN
AVAILABLE = (DIRECTORY / "model.pt").is_file()
if AVAILABLE:
    cfg = yaml.safe_load((DIRECTORY / "config.yaml").read_text())
    world, dataset_type, _ = build_world(cfg["dataset"])
    encoder = CNNEncoder(latent_dim=cfg["model"]["latent_dim"], image_size=world.image_size).eval()
    encoder.load_state_dict(torch.load(DIRECTORY / "model.pt", weights_only=True)["encoder"])
    history = torch.load(DIRECTORY / "train_history.pt", weights_only=False)
    exported = torch.load(DIRECTORY / "embeddings.pt", weights_only=False)
    final_epoch = max(exported["snapshots"])
    FIGURES = DIRECTORY / "figures"
    FIGURES.mkdir(exist_ok=True)
    print(f"{RUN}: using the model saved after epoch {final_epoch}.")
else:
    print(f"No completed model is saved for {RUN} yet. Select a completed run; dot_smoke is the short example.")

## Latent geometry

For the single dot, a regular grid of physical positions is encoded and
projected into PC space. Corresponding physical and latent points share
the same colors. Sprite and multi-dot worlds use saved evaluation samples;
multi-dot colors indicate the first dot's position.

The PCA basis and spectrum are fitted on a sample of training transitions
using the frozen encoder.

In [ ]:
if AVAILABLE:
    PCA_FIT_SAMPLES = min(10_000, int(cfg["dataset"]["train_samples"]))
    GRID_SIZE = 15
    fit_dataset = dataset_type(world, num_samples=PCA_FIT_SAMPLES, seed=int(cfg["training"]["seed"]))
    fit_loader = DataLoader(fit_dataset, batch_size=256, shuffle=False)
    chunks = []
    with torch.inference_mode():
        for before, _, after, _ in fit_loader:
            chunks.append(encoder(torch.cat([before, after])).numpy())
    z_fit = np.concatenate(chunks)
    total_variance = np.var(z_fit.astype(float), axis=0).sum()
    pca = PCA().fit(z_fit) if total_variance > 1e-24 else None
    variance = pca.explained_variance_ratio_ if pca is not None else np.zeros(z_fit.shape[1])

    sprite = cfg["dataset"].get("world", "structured") == "sprite"
    single_dot = not sprite and world.num_dots == 1
    if single_dot:
        from datasets.structured_dot_world import render_dots

        coordinates = np.linspace(world.dot_radius + 2, world.image_size - world.dot_radius - 3,
                                  GRID_SIZE, dtype=int)
        xx, yy = np.meshgrid(coordinates, coordinates)
        state = np.column_stack([xx.ravel(), yy.ravel()])
        images = np.stack([render_dots(xy[None], [world.groups[0].color_indices[0]], world) for xy in state])
        with torch.inference_mode():
            encoded = torch.cat([encoder(batch) for batch in torch.from_numpy(images).split(256)]).numpy()
    else:
        encoded = exported["snapshots"][final_epoch]["z_t"].numpy()
        state = exported["positions"].numpy().reshape(len(encoded), -1)[:, :2]
    projected = pca.transform(encoded) if pca is not None else np.zeros_like(encoded)
    colors = position_colors(state)

    # A circular orientation uses sine/cosine coordinates in the linear reference.
    reference_dim = world.action_dim + (int(world.control.mask[2]) if sprite else 0)
    fig, axes = plt.subplots(1, 3, figsize=(FULL_WIDTH, 2.75))
    fig.subplots_adjust(left=0.075, right=0.99, bottom=0.20, top=0.84, wspace=0.43)
    for ax in axes:
        ax.set_box_aspect(1)
        ax.xaxis.set_major_locator(MaxNLocator(4))
        ax.yaxis.set_major_locator(MaxNLocator(4))

    n = min(15, len(variance))
    axes[0].bar(np.arange(1, n + 1), variance[:n], width=0.55, color=BLUE)
    axes[0].axvline(reference_dim + 0.5, color=RED, ls="--", lw=0.9,
                    label=f"reference dim = {reference_dim}")
    axes[0].set(title="PCA spectrum", xlabel="Principal component",
                ylabel="Explained variance ratio", xlim=(0.3, n + 0.7))
    axes[0].set_xticks([tick for tick in [1, 5, 10, 15] if tick <= n])
    axes[0].set_ylim(0, max(0.05, float(variance[:n].max()) * 1.1))
    axes[0].legend(fontsize=7, loc="upper right", handlelength=1.1)
    axes[0].grid(axis="y")

    axes[1].scatter(*state.T, c=colors, s=13 if single_dot else 4,
                    linewidths=0, rasterized=True)
    axes[1].set(title="World state grid" if single_dot else "Physical positions",
                xlabel="x" if single_dot or sprite else "First dot x",
                ylabel="y" if single_dot or sprite else "First dot y")
    axes[1].set_aspect("equal", adjustable="box")

    axes[2].scatter(*projected[:, :2].T, c=colors, s=11 if single_dot else 4,
                    linewidths=0, rasterized=True)
    center = (projected[:, :2].min(axis=0) + projected[:, :2].max(axis=0)) / 2
    radius = max(float(np.ptp(projected[:, :2], axis=0).max()) * 0.56, 1e-6)
    axes[2].set(title="Encoded embeddings", xlabel="PC1", ylabel="PC2",
                xlim=(center[0] - radius, center[0] + radius),
                ylim=(center[1] - radius, center[1] + radius))
    axes[2].set_aspect("equal", adjustable="box")
    fig.savefig(FIGURES / "representation.pdf")
    plt.show()
    print(f"Mean training-embedding coordinate std: {np.std(z_fit, axis=0).mean():.4g}")

The red line marks the reference dimension: the controllable position coordinates,
with two coordinates for a controlled periodic orientation. The printed embedding
standard deviation complements the relative variance shown by the spectrum.

## Loss curves

Forward, inverse, and total losses on training and held-out samples.

In [ ]:
if AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(FULL_WIDTH, 2.15))
    fig.subplots_adjust(left=0.075, right=0.99, bottom=0.24, top=0.84, wspace=0.42)
    for ax, key, title in zip(axes, ["fwd", "inv", "total"], ["Forward loss", "Inverse loss", "Total loss"]):
        values = []
        for split, color in [("train", BLUE), ("eval", RED)]:
            curve = history[split]
            ax.plot(curve["epoch"], curve[key], color=color, label=split.capitalize(),
                    marker="o", markersize=2.5)
            values.extend(curve[key])
        if values and min(values) > 0:
            ax.set_yscale("log")
        ax.set(title=title, xlabel="Epoch")
        ax.grid(axis="y")
    axes[0].set_ylabel("Loss")
    axes[-1].legend()
    fig.savefig(FIGURES / "training_losses.pdf")
    plt.show()

## Frozen-encoder reconstruction

A post-hoc decoder visualizes information retained by the frozen encoder.
In the sprite experiments, compare reconstructions across action interfaces
to assess which aspects of position and orientation remain recoverable.

In [ ]:
if AVAILABLE and (DIRECTORY / "decoder.pt").is_file():
    dataset = dataset_type(world, num_samples=8,
                           seed=evaluation_seed(cfg["dataset"], cfg["training"]["seed"]))
    observations = torch.stack([dataset[i][0] for i in range(8)])
    decoder = CNNDecoder(latent_dim=cfg["model"]["latent_dim"], image_size=world.image_size).eval()
    decoder.load_state_dict(torch.load(DIRECTORY / "decoder.pt", weights_only=True))
    with torch.inference_mode():
        reconstructed = decoder(encoder(observations))
    fig, axes = plt.subplots(2, 8, figsize=(FULL_WIDTH, 2))
    fig.subplots_adjust(left=0.12, right=0.995, bottom=0.04, top=0.98, wspace=0.04, hspace=0.05)
    for i in range(8):
        axes[0, i].imshow(observations[i].permute(1, 2, 0).numpy(), vmin=0, vmax=1)
        axes[1, i].imshow(reconstructed[i].permute(1, 2, 0).numpy(), vmin=0, vmax=1)
        axes[0, i].set_axis_off()
        axes[1, i].set_axis_off()
    for row, label in enumerate(["Observation", "Reconstruction"]):
        box = axes[row, 0].get_position()
        fig.text(0.105, (box.y0 + box.y1) / 2, label, ha="right", va="center", fontsize=8)
    fig.savefig(FIGURES / "reconstructions.pdf")
    plt.show()
elif AVAILABLE:
    print("No decoder is attached to this run. See the sprite reconstruction experiment in the toy guide.")